In [0]:
%pip install aiohttp
%pip install aiolimiter

In [0]:
import os
import json
import time
import random
import aiohttp
import asyncio
import nest_asyncio
from threading import Lock
from aiolimiter import AsyncLimiter
from collections import Counter
from datetime import datetime, timezone

In [0]:
prod_url = "https://api.tibiadata.com/v4/"

base_dir="/Volumes/tibia_analytics/bronze/raw/"
headers = {
        "User-Agent": "Tibia Analysis version 0.0.1 (Educacional Project)"
    }

rate_limit = 32
limiter = AsyncLimiter(rate_limit, 1)

nest_asyncio.apply()

In [0]:
async def request(session, limiter, endpoint, retries=3):
    url = prod_url + endpoint

    for attempt in range(retries + 1):
        async with limiter:
            try:
                async with session.get(url) as response:
                    response.raise_for_status()
                    return await response.json()
            except aiohttp.ClientResponseError as e:
                if e.status in (429, 500, 502, 503, 504) and attempt < retries:
                    backoff = (2 ** attempt) + random.uniform(0.5, 1.5)
                    await asyncio.sleep(backoff)
                else:
                    raise
            except (aiohttp.ClientError, asyncio.TimeoutError):
                if attempt < retries:
                    backoff = (2 ** attempt) + random.uniform(0.5, 1.5)
                    await asyncio.sleep(backoff)
                else:
                    raise

In [0]:
def save_api_to_json(data, folder, filename):
    path = os.path.join(base_dir, folder)
    os.makedirs(path, exist_ok=True)

    filepath = os.path.join(path, filename)

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

In [0]:
async def save_async(data, folder, filename):
    await asyncio.to_thread(save_api_to_json, data, folder, filename)

In [0]:
async def get_worlds(session, limiter):
    return await request(session, limiter, "worlds")

In [0]:
async def get_highscores(session, limiter, world, category="experience", vocation="all", page=1):
    endpoint = f"highscores/{world}/{category}/{vocation}/{page}"
    return await request(session, limiter, endpoint)

In [0]:
async def get_characters(session, limiter, name):
    endpoint = f"character/{name}"
    return await request(session, limiter, endpoint)

In [0]:
async def get_highscores_world(session, limiter, world):
    first_page = await get_highscores(session, limiter, world, page=1)
    total_pages = first_page["highscores"]["highscore_page"]["total_pages"]

    tasks = [
        get_highscores(session, limiter, world, page=page)
        for page in range(2, total_pages + 1)
    ]

    results = await asyncio.gather(*tasks, return_exceptions=True)
    pages = [first_page]
    pages.extend(result for result in results if not isinstance(result, Exception))

    return pages

In [0]:
def get_world_names():
    worlds_df = spark.sql("""
      SELECT DISTINCT
             world.name AS world_name
        FROM tibia_analytics.bronze.raw_worlds
     LATERAL VIEW EXPLODE(worlds.regular_worlds) worlds_table AS world
       WHERE source_file_date = (
               SELECT MAX(source_file_date)
                 FROM tibia_analytics.bronze.raw_worlds
             )
    """)
    
    worlds = [row.world_name for row in worlds_df.collect()]
  
    if not worlds:
        raise RuntimeError(
            "No worlds found in bronze.raw_worlds for highscores ingestion."
        )

    return worlds

In [0]:
def get_characters_name():
    highscores_df = spark.sql("""
      SELECT DISTINCT
             highscore.world AS highscore_world,
             highscore.name AS highscore_name            
        FROM tibia_analytics.bronze.raw_highscores
     LATERAL VIEW EXPLODE(highscores.highscore_list) highscores_table AS highscore
       WHERE source_file_date = (
               SELECT MAX(source_file_date) 
                 FROM tibia_analytics.bronze.raw_highscores
             )
    """)
    highscores = [(row.highscore_world, row.highscore_name) for row in highscores_df.collect()]
  
    if not highscores:
        raise RuntimeError(
            "No characters found in bronze.raw_highscores for characters ingestion."
        )

    return highscores

In [0]:
async def run_worlds_ingestion():
    timeout = aiohttp.ClientTimeout(total=15, connect=5)
    connector = aiohttp.TCPConnector(limit=100, ttl_dns_cache=300)
    date_str = datetime.now(timezone.utc).strftime("%Y-%m-%d")
   
    async with aiohttp.ClientSession(headers=headers, timeout=timeout, connector=connector) as session:
        data = await get_worlds(session, limiter)
        await save_async(data, "worlds", f"worlds_{date_str}.json")

In [0]:
async def run_highscores_ingestion(world_list):
    timeout = aiohttp.ClientTimeout(total=15, connect=5)
    connector = aiohttp.TCPConnector(limit=100, ttl_dns_cache=300)
    date_str = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    async with aiohttp.ClientSession(headers=headers, timeout=timeout, connector=connector) as session:
        tasks = [
            get_highscores_world(session, limiter, world)
            for world in world_list
        ]
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        save_tasks = []

        for world, data in zip(world_list, results):
            if isinstance(data, Exception):
                print(f"Error fetching {world}: {data}")
                continue

            if data:
                save_tasks.append(
                    save_async(data, f"highscores/{date_str}", f"{world}.json")
                )

        if save_tasks:
            await asyncio.gather(*save_tasks)

In [0]:
async def get_world_characters(world_list=None):
    timeout = aiohttp.ClientTimeout(total=30, connect=5)
    connector = aiohttp.TCPConnector(limit=32, limit_per_host=32, ttl_dns_cache=300)
    date_str = datetime.now(timezone.utc).strftime("%Y-%m-%d")

    identity_df = spark.sql("""
      SELECT character_id,
             current_name,
             current_world,
             locked_names 
        FROM tibia_analytics.silver.characters_identity
       WHERE is_deleted = FALSE
          OR deleted_at > current_timestamp()
    """)

    tasks = []
    all_locked_names = set()

    for row in identity_df.collect():
        if row.locked_names:
            all_locked_names.update(row.locked_names)
        if world_list is None or row.current_world in world_list:
            tasks.append((row.character_id, row.current_name, row.current_world))

    world_names = get_characters_name()
    if world_list is None:
        world_list = {world for world, _ in world_names}

    for world_name, name in world_names:
        if world_name in world_list and name not in all_locked_names:
            tasks.append((None, name, world_name))

    results_by_world = {}

    async with aiohttp.ClientSession(headers=headers, timeout=timeout, connector=connector) as session:
        async def worker(queue: asyncio.Queue):

            while True:
                item = await queue.get()
                if item is None:
                    queue.task_done()
                    break

                character_id, name, world = item

                try:
                    response = await get_characters(session, limiter, name)
                    if response is not None:
                        real_world = response.get("character", {}).get("character", {}).get("world", world)
                        results_by_world.setdefault(real_world, []).append(response)

                except Exception as e:
                    print(f"Error {name}: {e}")

                finally:
                    queue.task_done()
        
        queue = asyncio.Queue()
        for task in tasks:
            queue.put_nowait(task)

        num_workers = 32
        workers = [asyncio.create_task(worker(queue)) for _ in range(num_workers)]

        await queue.join()

        for _ in range(num_workers):
            queue.put_nowait(None)

        await asyncio.gather(*workers)

    save_character_tasks = [
        save_async(results, f"characters/{date_str}", f"{world}.json")
        for world, results in results_by_world.items()
    ]

    await asyncio.gather(*save_character_tasks)